## Ingest circuits.csv file

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze-helpers

In [0]:
source_file = f"{landing_folder_path}/circuits.csv"
table_name = f"{catalog_name}.{bronze_schema}.circuits"

### 1 - Read the CSV using the dataframe reader API

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

circuits_schema = StructType([
    StructField('circuitId' ,   StringType()),
    StructField('url',          StringType()),
    StructField('circuitName',  StringType()),
    StructField('lat',          DoubleType()),
    StructField('long',         DoubleType()),
    StructField('locality',     StringType()),
    StructField('country',      StringType()),
])

In [0]:
circuits_df = (
    spark.read
    .format('csv')
    .option('header', 'true')
    .option('mode', 'FAILFAST')
    # .option('inferschema', 'true)
    .schema(circuits_schema)
    .load(source_file)
)

In [0]:
circuits_df.show(truncate=False)

In [0]:
display(circuits_df)

### 2 - Add Metadata Columns -> Source File | Ingestion Timestamp

In [0]:
circuits_final_df = add_ingestion_metadata(circuits_df)

In [0]:
display(circuits_final_df)

### 3 - Write to bronze delta table

In [0]:
(
    circuits_final_df
    .write
    .format('delta')
    .mode('overwrite')
    .saveAsTable(table_name)
)

In [0]:
%sql

SELECT * FROM formula1ext.bronze.circuits;

In [0]:
spark.table('formula1ext.bronze.circuits').display()